# PTB-XL Preprocessing

This notebook prepares metadata, audits patient-level splits, materializes unfiltered ECG waveforms after record-wise/lead-wise normalization, and prepares the two clinical inputs for a multimodal model.

- Map `scp_codes` to the five diagnostic superclasses and quarantine form/rhythm-only patient groups for modeling review.
- Audit patient/ECG isolation; fit clinical age mean/std using observed training records only.
- Do not apply signal filters. Normalize each ECG record independently, lead by lead, using z-scores; replace non-finite normalized values with zero.
- Keep clinical inputs to `age` and `sex`: z-score age with train-only statistics (missing ages map to the training mean, hence z-score 0) and preserve sex as binary 0/1.
- Save all ECG and clinical records, including validation/test and quarantined records, with indexes keyed by `ecg_id`. This is preprocessing only, not model training.

In [ ]:
from pathlib import Path
import ast
import json
import pandas as pd
import numpy as np
import wfdb

current_dir = Path.cwd().resolve()
project_root = next(
    (candidate for candidate in (current_dir, *current_dir.parents)
     if (candidate / 'data' / 'ptbxl_database.csv').is_file()),
    None,
)
if project_root is None:
    raise FileNotFoundError('Could not locate data/ptbxl_database.csv from this notebook working directory.')
data_dir = project_root / 'data'

SUPERCLASSES = ['NORM', 'MI', 'STTC', 'CD', 'HYP']


def load_metadata(data_dir):
    return pd.read_csv(Path(data_dir) / 'ptbxl_database.csv', encoding='latin-1')


def load_scp_statements(data_dir):
    return pd.read_csv(Path(data_dir) / 'scp_statements.csv', index_col=0)


def parse_scp_codes(value):
    if pd.isna(value):
        return {}
    try:
        parsed = ast.literal_eval(value)
    except (TypeError, ValueError, SyntaxError):
        return {}
    return parsed if isinstance(parsed, dict) else {}


def build_superclass_lookup(scp_statements):
    diagnostic = scp_statements.loc[scp_statements['diagnostic'].eq(1.0)]
    return {
        str(code): str(row['diagnostic_class'])
        for code, row in diagnostic.iterrows()
        if pd.notna(row.get('diagnostic_class')) and str(row['diagnostic_class']) in SUPERCLASSES
    }


def mapping_from_scp_codes(series, superclass_lookup):
    rows = []
    for value in series:
        labels = {superclass_lookup[code] for code in parse_scp_codes(value) if code in superclass_lookup}
        rows.append({label: int(label in labels) for label in SUPERCLASSES})
    return pd.DataFrame(rows, index=series.index, columns=SUPERCLASSES)


def get_patient_split(metadata):
    if 'patient_id' not in metadata or 'strat_fold' not in metadata:
        raise ValueError('Metadata must contain patient_id and strat_fold.')
    folds = sorted(metadata['strat_fold'].dropna().unique())
    if len(folds) >= 10:
        train_folds, val_fold, test_fold = [fold for fold in folds if fold not in (9, 10)], 9, 10
    elif len(folds) >= 3:
        train_folds, val_fold, test_fold = folds[:-2], folds[-2], folds[-1]
    else:
        raise ValueError('At least three strat_fold values are required.')
    train = metadata.loc[metadata['strat_fold'].isin(train_folds)].copy()
    validation = metadata.loc[metadata['strat_fold'].eq(val_fold)].copy()
    test = metadata.loc[metadata['strat_fold'].eq(test_fold)].copy()
    return train, validation, test


def load_waveform(data_dir, filename, expected_sampling_rate=500):
    if pd.isna(filename):
        raise ValueError('A non-empty PTB-XL waveform filename is required.')
    record_path = Path(filename)
    if record_path.suffix:
        record_path = record_path.with_suffix('')
    if not record_path.is_absolute():
        record_path = Path(data_dir) / record_path
    header_path = record_path.with_suffix('.hea')
    if not header_path.is_file():
        raise FileNotFoundError(f'WFDB header not found: {header_path}')
    signal, fields = wfdb.rdsamp(str(record_path))
    if fields.get('fs') != expected_sampling_rate:
        raise ValueError(f"Expected {expected_sampling_rate} Hz, found {fields.get('fs')} Hz for {filename}.")
    if signal.ndim != 2:
        raise ValueError(f'Expected a 2D (samples, leads) signal, got shape {signal.shape}.')
    return np.asarray(signal, dtype=np.float32), fields


meta = load_metadata(data_dir)
print(meta.shape)
print(meta.head(3).to_string(index=False))

In [ ]:
scp_statements = load_scp_statements(data_dir)
superclass_lookup = build_superclass_lookup(scp_statements)
print('Superclass mapping sample:')
print(list(superclass_lookup.items())[:10])

labels_df = mapping_from_scp_codes(meta['scp_codes'], superclass_lookup)
print('\nLabel frame shape:', labels_df.shape)
print(labels_df.head().to_string(index=False))
print('\nSuperclass frequency:')
print(labels_df[SUPERCLASSES].sum().to_string())

In [ ]:
meta = meta.join(labels_df)
train_df, val_df, test_df = get_patient_split(meta)
rows_before_quarantine = (len(train_df), len(val_df), len(test_df))

report_title = 'ECG cần rà soát: SCP form/rhythm chưa ánh xạ superclass'
form_rhythm_codes = {
    str(code)
    for code, row in scp_statements.iterrows()
    if row.get('form') == 1.0 or row.get('rhythm') == 1.0
}

def is_form_rhythm_only_unmapped(value):
    code_names = set(parse_scp_codes(value))
    return (
        bool(code_names)
        and code_names.issubset(form_rhythm_codes)
        and code_names.isdisjoint(superclass_lookup)
    )

unmapped_form_rhythm_mask = meta['scp_codes'].map(is_form_rhythm_only_unmapped)
unmapped_form_rhythm_records = meta.loc[unmapped_form_rhythm_mask].copy()
if unmapped_form_rhythm_records['patient_id'].isna().any():
    raise ValueError('Cannot quarantine matching records with missing patient_id.')
review_patient_ids = set(unmapped_form_rhythm_records['patient_id'])
quarantine_records = meta.loc[meta['patient_id'].isin(review_patient_ids)].copy()

train_df = train_df.loc[~train_df['patient_id'].isin(review_patient_ids)].copy()
val_df = val_df.loc[~val_df['patient_id'].isin(review_patient_ids)].copy()
test_df = test_df.loc[~test_df['patient_id'].isin(review_patient_ids)].copy()
train_patients = set(train_df['patient_id'])
val_patients = set(val_df['patient_id'])
test_patients = set(test_df['patient_id'])
split_patient_sets = (train_patients, val_patients, test_patients)
split_ecg_ids = set(train_df['ecg_id']) | set(val_df['ecg_id']) | set(test_df['ecg_id'])

print('=' * len(report_title))
print(report_title)
print('=' * len(report_title))
print('Matching ECG records:', len(unmapped_form_rhythm_records))
print('Patients reserved:', len(review_patient_ids))
print('All ECG records reserved by patient:', len(quarantine_records))
print('Train/val/test before quarantine:', rows_before_quarantine)
print('Train/val/test after quarantine:', len(train_df), len(val_df), len(test_df))
print('Train/val/test patients after quarantine:', len(train_patients), len(val_patients), len(test_patients))
print('Patient overlap:', sum(len(split_patient_sets[i] & split_patient_sets[j]) for i in range(3) for j in range(i + 1, 3)))
print('Split/quarantine prepared; independent audit runs in the next cell.')


In [ ]:
# Independent audit: report split leakage explicitly and stop Run All on failure.
from itertools import combinations

audit_partitions = {
    'train': train_df,
    'validation': val_df,
    'test': test_df,
    'quarantine': quarantine_records,
}
audit_issues = []
if meta['patient_id'].isna().any():
    audit_issues.append(f'Source metadata has {int(meta["patient_id"].isna().sum())} missing patient_id value(s).')
if not meta['ecg_id'].is_unique:
    audit_issues.append('Source metadata contains duplicate ecg_id values.')
audit_patient_ids = {name: set(frame['patient_id'].dropna()) for name, frame in audit_partitions.items()}
audit_ecg_ids = {name: set(frame['ecg_id'].dropna()) for name, frame in audit_partitions.items()}

for left_name, right_name in combinations(audit_partitions, 2):
    shared_patients = audit_patient_ids[left_name] & audit_patient_ids[right_name]
    shared_ecgs = audit_ecg_ids[left_name] & audit_ecg_ids[right_name]
    if shared_patients:
        examples = sorted(shared_patients, key=str)[:5]
        audit_issues.append(f'Patient overlap {left_name}/{right_name}: {len(shared_patients)}; examples={examples}.')
    if shared_ecgs:
        examples = sorted(shared_ecgs, key=str)[:5]
        audit_issues.append(f'ECG overlap {left_name}/{right_name}: {len(shared_ecgs)}; examples={examples}.')

all_audit_ecg_ids = set().union(*audit_ecg_ids.values())
all_audit_patient_ids = set().union(*audit_patient_ids.values())
source_ecg_ids = set(meta['ecg_id'].dropna())
source_patient_ids = set(meta['patient_id'].dropna())
if all_audit_ecg_ids != source_ecg_ids:
    audit_issues.append(f'ECG coverage mismatch: missing={len(source_ecg_ids - all_audit_ecg_ids)}, extra={len(all_audit_ecg_ids - source_ecg_ids)}.')
if all_audit_patient_ids != source_patient_ids:
    audit_issues.append(f'Patient coverage mismatch: missing={len(source_patient_ids - all_audit_patient_ids)}, extra={len(all_audit_patient_ids - source_patient_ids)}.')
partition_record_count = sum(len(frame) for frame in audit_partitions.values())
if partition_record_count != len(meta):
    audit_issues.append(f'Record count mismatch: partitions={partition_record_count}, source={len(meta)}.')
if set(train_df['strat_fold'].dropna()) != set(range(1, 9)):
    audit_issues.append('Train fold assignment is not exactly folds 1-8.')
if set(val_df['strat_fold'].dropna()) != {9}:
    audit_issues.append('Validation fold assignment is not exactly fold 9.')
if set(test_df['strat_fold'].dropna()) != {10}:
    audit_issues.append('Test fold assignment is not exactly fold 10.')

if audit_issues:
    print('SPLIT AUDIT FAILED: leakage or partition inconsistency detected')
    for issue in audit_issues:
        print('-', issue)
    raise AssertionError('Resolve split audit findings before continuing to preprocessing or model evaluation.')
else:
    print('SPLIT AUDIT PASSED: no patient/ECG overlap detected.')
    print('Records (train/validation/test/quarantine):', *(len(frame) for frame in audit_partitions.values()))
    print('Patients (train/validation/test/quarantine):', *(len(audit_patient_ids[name]) for name in audit_partitions))
    print('All source records and patients are covered exactly once; folds 1-8/9/10 are preserved.')


In [ ]:
# Prepare clinical inputs for the multimodal model: age (train-only z-score) and binary sex.
clinical_columns = ['age', 'sex']
missing_clinical_columns = set(clinical_columns) - set(meta.columns)
if missing_clinical_columns:
    raise ValueError(f'Missing required clinical columns: {sorted(missing_clinical_columns)}')
if meta['ecg_id'].isna().any() or meta['ecg_id'].duplicated().any():
    raise ValueError('Metadata ecg_id values must be present and unique.')

train_age = pd.to_numeric(train_df['age'], errors='raise')
observed_train_age = train_age.dropna().to_numpy(dtype=np.float64)
if observed_train_age.size == 0 or not np.isfinite(observed_train_age).all():
    raise ValueError('Training age must contain finite observed values to fit z-score statistics.')
age_train_mean = float(observed_train_age.mean())
age_train_std = float(observed_train_age.std(ddof=0))
if not np.isfinite(age_train_std) or age_train_std <= 0:
    raise ValueError('Training age standard deviation must be positive and finite.')

all_age = pd.to_numeric(meta['age'], errors='raise')
if not np.isfinite(all_age.dropna().to_numpy(dtype=np.float64)).all():
    raise ValueError('Non-finite non-missing age values were found.')
age_missing_mask = all_age.isna()
age_imputed = all_age.fillna(age_train_mean)
age_z = (age_imputed - age_train_mean) / age_train_std

all_sex = pd.to_numeric(meta['sex'], errors='raise')
if all_sex.isna().any():
    raise ValueError('Sex contains missing values; define an explicit missing-value policy before training.')
if not all_sex.isin([0, 1]).all():
    raise ValueError('Sex must be encoded as 0/1 in the PTB-XL metadata.')

clinical_features = pd.DataFrame({
    'ecg_id': meta['ecg_id'].to_numpy(),
    'age_z': age_z.to_numpy(dtype=np.float32),
    'sex': all_sex.to_numpy(dtype=np.float32),
}).set_index('ecg_id')
if not np.isfinite(clinical_features.to_numpy()).all():
    raise ValueError('Prepared clinical features contain non-finite values.')

clinical_feature_cols = ['age_z', 'sex']
X_train_clinical = clinical_features.loc[train_df['ecg_id'], clinical_feature_cols].to_numpy(dtype=np.float32)
X_val_clinical = clinical_features.loc[val_df['ecg_id'], clinical_feature_cols].to_numpy(dtype=np.float32)
X_test_clinical = clinical_features.loc[test_df['ecg_id'], clinical_feature_cols].to_numpy(dtype=np.float32)

# Mục tiêu đa nhãn gồm 5 superclass: NORM, MI, STTC, CD, HYP.
y_train = train_df[SUPERCLASSES].to_numpy(dtype=np.float32)
y_val = val_df[SUPERCLASSES].to_numpy(dtype=np.float32)
y_test = test_df[SUPERCLASSES].to_numpy(dtype=np.float32)

processed_dir = data_dir / 'processed'
processed_dir.mkdir(parents=True, exist_ok=True)
clinical_index_path = processed_dir / 'ptbxl_clinical_features_age_zscore_sex_binary.csv'
clinical_stats_path = processed_dir / 'ptbxl_clinical_normalization_stats.json'
clinical_features.reset_index().to_csv(clinical_index_path, index=False)
with clinical_stats_path.open('w', encoding='utf-8') as stats_file:
    json.dump({
        'age_mean_train': age_train_mean,
        'age_std_train_ddof0': age_train_std,
        'age_missing_policy': 'impute_with_train_mean_before_zscore',
        'sex_encoding': {'0': 0, '1': 1},
        'feature_order': clinical_feature_cols,
    }, stats_file, indent=2)

assert X_train_clinical.shape == (len(train_df), 2)
assert X_val_clinical.shape == (len(val_df), 2)
assert X_test_clinical.shape == (len(test_df), 2)
assert np.isfinite(X_train_clinical).all() and np.isfinite(X_val_clinical).all() and np.isfinite(X_test_clinical).all()
assert np.isin(np.concatenate([X_train_clinical[:, 1], X_val_clinical[:, 1], X_test_clinical[:, 1]]), [0, 1]).all()

print('Clinical feature order:', clinical_feature_cols)
print('Age z-score mean/std fit on observed train rows:', f'{age_train_mean:.6f}', f'{age_train_std:.6f}')
print('Age missing, imputed with train mean:', {
    'train': int(train_df['age'].isna().sum()),
    'validation': int(val_df['age'].isna().sum()),
    'test': int(test_df['age'].isna().sum()),
    'all_metadata': int(age_missing_mask.sum()),
})
print('Sex encoding:', sorted(all_sex.unique().tolist()), '| missing:', int(all_sex.isna().sum()))
print('Train/validation/test clinical shapes:', X_train_clinical.shape, X_val_clinical.shape, X_test_clinical.shape)
print('Train/validation/test target shapes:', y_train.shape, y_val.shape, y_test.shape)
print('Saved clinical features:', clinical_index_path)
print('Saved train-only normalization statistics:', clinical_stats_path)

## Unfiltered record-wise, lead-wise normalization

The following cells replace the earlier baseline, notch, and wavelet filtering previews. They apply z-score normalization independently to each lead of one ECG record, using finite samples for that lead's mean and standard deviation; missing/non-finite inputs become zero in the normalized signal.

In [ ]:
def record_leadwise_zscore(signal, epsilon=1e-8):
    """Z-score each lead using finite samples from this record; map invalid values to zero."""
    array = np.asarray(signal, dtype=np.float64)
    if array.ndim != 2 or array.shape[0] == 0 or array.shape[1] == 0:
        raise ValueError('Expected a non-empty 2D (samples, leads) ECG array.')
    if not np.isfinite(epsilon) or epsilon <= 0:
        raise ValueError('epsilon must be a positive finite number.')

    normalized = np.zeros_like(array, dtype=np.float64)
    for lead_index in range(array.shape[1]):
        lead = array[:, lead_index]
        valid = np.isfinite(lead)
        if valid.any():
            values = lead[valid]
            mean = values.mean()
            standard_deviation = values.std()
            normalized[valid, lead_index] = (values - mean) / (standard_deviation + epsilon)
    return np.nan_to_num(normalized, copy=False, nan=0.0, posinf=0.0, neginf=0.0)


# Check finite, missing, infinite, and all-missing lead behavior.
normalization_check_input = np.array([
    [1.0, np.nan, np.inf],
    [2.0, np.inf, np.nan],
    [3.0, 7.0, np.nan],
])
normalization_check = record_leadwise_zscore(normalization_check_input)
assert np.isfinite(normalization_check).all()
assert np.all(normalization_check[~np.isfinite(normalization_check_input)] == 0.0)
assert np.isclose(normalization_check[:, 0].mean(), 0.0, atol=1e-12)
assert np.isclose(normalization_check[:, 0].std(), 1.0, atol=1e-7)
assert np.all(normalization_check[:, 2] == 0.0)
print('Synthetic normalization checks passed, including missing/non-finite values mapped to zero.')

In [ ]:
import matplotlib.pyplot as plt

standard_leads = ['I', 'II', 'III', 'AVR', 'AVL', 'AVF', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6']
normalization_candidates = train_df.sort_values('ecg_id')
if normalization_candidates.empty:
    raise ValueError('No ECG records found in the training split.')
normalization_row = normalization_candidates.iloc[0]
loaded_signal, normalization_fields = load_waveform(data_dir, normalization_row['filename_hr'])
if normalization_fields['fs'] != 500:
    raise ValueError(f"Expected 500 Hz, got {normalization_fields['fs']} Hz.")
if set(normalization_fields['sig_name']) != set(standard_leads):
    raise ValueError(f"Expected the standard 12 leads, got {normalization_fields['sig_name']}.")
lead_order = [normalization_fields['sig_name'].index(lead) for lead in standard_leads]
raw_signal = loaded_signal[:, lead_order].astype(np.float64, copy=True)
normalized_signal = record_leadwise_zscore(raw_signal)
assert raw_signal.shape == normalized_signal.shape
assert raw_signal.shape[1] == 12
assert np.isfinite(normalized_signal).all()

input_nonfinite_count = int(np.count_nonzero(~np.isfinite(raw_signal)))
actual_labels = [label for label in SUPERCLASSES if normalization_row[label] == 1]
time_seconds = np.arange(raw_signal.shape[0]) / normalization_fields['fs']
fig_normalized, axes_normalized = plt.subplots(12, 1, figsize=(14, 18), sharex=True)
for lead_index, (axis, lead_name) in enumerate(zip(axes_normalized, standard_leads)):
    axis.plot(time_seconds, normalized_signal[:, lead_index], color='#167d9a', linewidth=0.65)
    axis.set_ylabel(f'{lead_name}\nz-score', rotation=0, labelpad=30, va='center')
    axis.grid(True, alpha=0.25)
axes_normalized[-1].set_xlabel('Time (s)')
duration_seconds = raw_signal.shape[0] / normalization_fields['fs']
fig_normalized.suptitle(
    f"ECG {int(normalization_row['ecg_id'])} | patient {int(normalization_row['patient_id'])} | "
    f"{', '.join(actual_labels) or 'no mapped superclass'} | 500 Hz | {duration_seconds:.1f} s"
)
fig_normalized.tight_layout(rect=(0, 0, 1, 0.98))
plt.show()
print('Record:', normalization_row['filename_hr'])
print('ECG ID / patient ID / labels:', int(normalization_row['ecg_id']), int(normalization_row['patient_id']), actual_labels)
print('Raw and normalized shape:', raw_signal.shape, normalized_signal.shape)
print('Lead order:', standard_leads)
print('Sampling rate / duration:', normalization_fields['fs'], 'Hz /', f'{duration_seconds:.1f} s')
print('Input NaN/Inf values replaced with zero after normalization:', input_nonfinite_count)
print('Normalization: independent mean/std per record and lead; epsilon = 1e-8; no filtering or resampling.')

In [ ]:
import os
import shutil

# Materialize normalized signals for every metadata record, including reserved records.
ordered_records = meta.sort_values('ecg_id').reset_index(drop=True)
if ordered_records['ecg_id'].isna().any() or ordered_records['ecg_id'].duplicated().any():
    raise ValueError('Metadata ecg_id values must be present and unique.')
if ordered_records['filename_hr'].isna().any():
    raise ValueError('Every metadata record must have a high-resolution waveform filename.')

expected_samples = 5000
n_records = len(ordered_records)
output_shape = (n_records, expected_samples, len(standard_leads))
required_bytes = int(np.prod(output_shape, dtype=np.int64)) * np.dtype(np.float32).itemsize
output_dir = data_dir / 'processed'
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / 'ptbxl_recordwise_leadwise_zscore_500hz.npy'
index_path = output_dir / 'ptbxl_recordwise_leadwise_zscore_500hz_index.csv'
temporary_output_path = output_path.with_name(output_path.stem + '.tmp.npy')
temporary_index_path = index_path.with_name(index_path.name + '.tmp')
free_bytes = shutil.disk_usage(output_dir).free
if free_bytes < required_bytes + 256 * 1024**2:
    raise OSError(
        f'Insufficient free disk space: need about {required_bytes / 1024**3:.2f} GiB '
        'plus a 256 MiB safety margin.'
    )

normalized_store = np.lib.format.open_memmap(
    temporary_output_path, mode='w+', dtype=np.float32, shape=output_shape
)
nonfinite_input_count = 0
try:
    for position, row in ordered_records.iterrows():
        signal, fields = load_waveform(data_dir, row['filename_hr'])
        if fields['fs'] != 500:
            raise ValueError(f"Expected 500 Hz for ECG {row['ecg_id']}, got {fields['fs']} Hz.")
        if signal.shape != (expected_samples, len(standard_leads)):
            raise ValueError(
                f"Expected {(expected_samples, len(standard_leads))} for ECG {row['ecg_id']}, "
                f'got {signal.shape}.'
            )
        if set(fields['sig_name']) != set(standard_leads):
            raise ValueError(f"Unexpected lead names for ECG {row['ecg_id']}: {fields['sig_name']}.")
        lead_order = [fields['sig_name'].index(lead) for lead in standard_leads]
        signal = signal[:, lead_order]
        nonfinite_input_count += int(np.count_nonzero(~np.isfinite(signal)))
        normalized = record_leadwise_zscore(signal)
        if not np.isfinite(normalized).all():
            raise ValueError(f"Non-finite normalized values remain for ECG {row['ecg_id']}.")
        normalized_store[position] = normalized.astype(np.float32, copy=False)
        if (position + 1) % 1000 == 0 or position + 1 == n_records:
            print(f'Normalized {position + 1:,}/{n_records:,} records.')

    normalized_store.flush()
    del normalized_store
    ordered_records.loc[:, ['ecg_id', 'patient_id', 'strat_fold', 'filename_hr'] + SUPERCLASSES].assign(
        array_position=np.arange(n_records, dtype=np.int64)
    ).to_csv(temporary_index_path, index=False)
    os.replace(temporary_output_path, output_path)
    os.replace(temporary_index_path, index_path)
except Exception:
    if 'normalized_store' in locals():
        del normalized_store
    for temporary_path in (temporary_output_path, temporary_index_path):
        if temporary_path.exists():
            temporary_path.unlink()
    raise

saved_signals = np.load(output_path, mmap_mode='r')
index_df = pd.read_csv(index_path)
if saved_signals.shape != output_shape or len(index_df) != n_records:
    raise ValueError('Saved normalized signal array and index failed shape/count validation.')
if not np.isfinite(saved_signals).all():
    raise ValueError('Saved normalized signal array contains non-finite values.')
if not np.array_equal(index_df['ecg_id'].to_numpy(), ordered_records['ecg_id'].to_numpy()):
    raise ValueError('Saved index order does not match normalized array order.')
print('All metadata records normalized, including validation/test and quarantined records.')
print('Signal array:', output_path, '| shape:', saved_signals.shape, '| dtype:', saved_signals.dtype)
print('Index:', index_path, '| records:', len(index_df))
print('Lead order:', standard_leads, '| sampling rate: 500 Hz | samples per record:', expected_samples)
print('Input NaN/Inf values mapped to zero:', nonfinite_input_count)
print('Output size:', f'{output_path.stat().st_size / 1024**3:.2f} GiB')

## Kết luận nhanh

- Metadata được chia fold và audit tách biệt patient; age được z-score bằng mean/std chỉ fit từ age quan sát được trong train (ddof=0).
- Clinical input theo đúng thiết kế hiện tại chỉ gồm `[age_z, sex]`: age thiếu được thay bằng train mean trước chuẩn hóa (z-score 0); sex giữ mã nhị phân 0/1. Không dùng height/weight.
- Các ma trận `X_train_clinical`, `X_val_clinical`, `X_test_clinical` có thứ tự cột `[age_z, sex]`; nhãn tương ứng là 5 superclass.
- Clinical features toàn bộ metadata được lưu ở `data/processed/ptbxl_clinical_features_age_zscore_sex_binary.csv`; thông số train-only lưu ở `data/processed/ptbxl_clinical_normalization_stats.json`.
- Không áp dụng baseline correction, notch hay wavelet filtering; ECG giữ 500 Hz, 5.000 mẫu × 12 lead. Đã chuẩn hóa toàn bộ 21.837 ECG record-wise, lead-wise và lưu mảng float32 cùng index ở `data/processed/`.
- Một lead hằng được giữ thành 0: ECG 12722, V5 có waveform gốc toàn 0. Bước tiếp theo là nối ECG và clinical batches theo `ecg_id`/`array_position` trong Dataset/DataLoader; chưa có code huấn luyện mô hình.